In [107]:
from typing import Optional
from datetime import datetime
import pandas as pd
import numpy as np
import nfl_data_py as nfl

pd.set_option('display.max_columns', None)

# from data.schema.database import (
#     Player, NFLSeasonStats, NFLAdvancedStats,
#     NFLWeeklySnaps, InjuryRecord, get_session
# )

# Positions we care about for fantasy
SKILL_POSITIONS = {"QB", "RB", "WR", "TE"}

# Injury severity mapping — used for composite injury risk score
INJURY_SEVERITY = {
    "Knee": 0.8, "ACL": 1.0, "MCL": 0.7, "Meniscus": 0.7,
    "Hamstring": 0.6, "Quad": 0.5, "Groin": 0.5, "Hip": 0.5,
    "Ankle": 0.6, "Foot": 0.5,
    "Shoulder": 0.6, "Clavicle": 0.5, "Elbow": 0.4, "Wrist": 0.4,
    "Back": 0.7, "Ribs": 0.5,
    "Concussion": 0.8,
    "Illness": 0.2, "Rest": 0.0, "Not Injury Related": 0.0,
}

# Soft tissue injuries have higher recurrence risk
SOFT_TISSUE_INJURIES = {"Hamstring", "Quad", "Groin", "Hip", "Calf", "Thigh"}

In [126]:
def run_full_backfill(start_year: int = 2015, end_year: Optional[int] = None):
        """
        Master method — runs all ingestion steps in order.
        Pulls everything from start_year to present.
        Safe to re-run (upserts, not inserts).
        """
        if not end_year:
            end_year = datetime.now().year
        years = list(range(start_year, end_year + 1))

        print(f"Starting NFL backfill for {years[0]}–{years[-1]}")

        ingest_players()
        ingest_seasonal_stats(years)
        ingest_advanced_stats(years)
        ingest_snap_counts(years)
        ingest_injuries(years)

        print("NFL backfill complete.")

def _load_snap_pct_for_season_stats(years: list[int]) -> pd.DataFrame:
        """Aggregate snap % per player per season from weekly snap data."""
        try:
            snaps = nfl.import_snap_counts(years)
            snap_agg = (
                snaps[snaps["position"].isin(SKILL_POSITIONS)]
                .groupby(["pfr_player_id", "season"])
                .agg(snap_pct=("offense_pct", "mean"))
                .reset_index()
                .rename(columns={"pfr_player_id": "player_id"})
            )
            return snap_agg
        except Exception as e:
            print(f"Could not load snap data: {e}")
            return pd.DataFrame()
    
def _load_team_targets(years: list[int]) -> pd.DataFrame:
    """Compute team-level target totals (needed for target_share)."""
    try:
        pbp = nfl.import_pbp_data(years, columns=[
            "passer_player_id", "receiver_player_id", "pass_attempt",
            "posteam", "season", "play_type"
        ])
        pass_plays = pbp[pbp["pass_attempt"] == 1]
        team_targets = (
            pass_plays.groupby(["posteam", "season"])
            .size()
            .reset_index(name="team_target_total")
        )
        return team_targets
    except Exception as e:
        print(f"Could not load team targets: {e}")
        return pd.DataFrame()
            
def ingest_seasonal_stats(years: list[int]):
        """
        Pulls seasonal aggregates: standard stats + fantasy points + efficiency.
        This is the core stats table.
        """
        print(f"Ingesting seasonal stats for {years}...")

        player_stats_df = nfl.import_seasonal_data(years, s_type="REG")
        # print("RAW DATA")
        # print(type(player_stats_df))
        # print(player_stats_df.columns)
        snap_df = _load_snap_pct_for_season_stats(years)
        # print("SNAP DATA")
        # print(type(snap_df))
        # print(snap_df.columns)
        team_targets = _load_team_targets(years)
        # print("TEAM TARGETS DATA")
        # print(type(team_targets))
        # print(team_targets.columns)
        
        # Merge snap pct and team targets into seasonal
        if not snap_df.empty:
            player_stats_df = player_stats_df.merge(snap_df, on=["player_id", "season"], how="left")
        if not team_targets.empty:
            player_stats_df = player_stats_df.merge(team_targets, on=["season"], how="left")

        # Compute derived metrics not in raw data
        player_stats_df["wopr"] = (1.5 * player_stats_df.get("target_share", 0)) + (0.7 * player_stats_df.get("air_yards_share", 0))
        player_stats_df["tgt_per_game"] = player_stats_df["targets"] / player_stats_df["games"].replace(0, float("nan"))
        player_stats_df["catch_rate"] = player_stats_df["receptions"] / player_stats_df["targets"].replace(0, float("nan"))
        player_stats_df["fantasy_ppg_ppr"] = player_stats_df["fantasy_points_ppr"] / player_stats_df["games"].replace(0, float("nan"))
        player_stats_df["fantasy_ppg_half"] = (player_stats_df.get("fantasy_points", player_stats_df["fantasy_points_ppr"] * 0.9)) / player_stats_df["games"].replace(0, float("nan"))
        player_stats_df["completion_pct"] = player_stats_df["completions"] / player_stats_df["attempts"].replace(0, float("nan"))

        records = []
        return player_stats_df
        # df = pd.DataFrame()
        # for _, row in player_stats_df.iterrows():
        #     # break
        #     print(row.columns)

        #     df_row = pd.DataFrame(
        #         [
        #             dict(
        #                 player_id=str(row["player_id"]),
        #                 season=int(row["season"]),
        #                 season_type="REG",
        #                 team=row.get("recent_team"),
        #                 games=_safe_int(row.get("games")),
        #                 games_started=_safe_int(row.get("games_started")),
        #                 completions=_safe_int(row.get("completions")),
        #                 attempts=_safe_int(row.get("attempts")),
        #                 passing_yards=_safe_int(row.get("passing_yards")),
        #                 passing_tds=_safe_int(row.get("passing_tds")),
        #                 interceptions=_safe_int(row.get("interceptions")),
        #                 passing_epa=row.get("passing_epa"),
        #                 completion_pct=row.get("completion_pct"),
        #                 yards_per_attempt=row.get("yards_per_attempt"),
        #                 passer_rating=row.get("passer_rating"),
        #                 sacks=_safe_int(row.get("sacks")),
        #                 carries=_safe_int(row.get("carries")),
        #                 rushing_yards=_safe_int(row.get("rushing_yards")),
        #                 rushing_tds=_safe_int(row.get("rushing_tds")),
        #                 rushing_epa=row.get("rushing_epa"),
        #                 yards_per_carry=row.get("rushing_yards_per_att"),
        #                 targets=_safe_int(row.get("targets")),
        #                 receptions=_safe_int(row.get("receptions")),
        #                 receiving_yards=_safe_int(row.get("receiving_yards")),
        #                 receiving_tds=_safe_int(row.get("receiving_tds")),
        #                 receiving_epa=row.get("receiving_epa"),
        #                 yards_per_reception=row.get("yards_per_reception"),
        #                 catch_rate=row.get("catch_rate"),
        #                 yards_per_target=row.get("yards_per_target"),
        #                 air_yards_total=_safe_int(row.get("receiving_air_yards")),
        #                 yards_after_catch=_safe_int(row.get("receiving_yards_after_catch")),
        #                 fantasy_points_ppr=row.get("fantasy_points_ppr"),
        #                 fantasy_ppg_ppr=row.get("fantasy_ppg_ppr"),
        #                 target_share=row.get("target_share"),
        #                 air_yards_share=row.get("air_yards_share"),
        #                 racr=row.get("racr"),
        #                 wopr=row.get("wopr"),
        #                 tgt_per_game=row.get("tgt_per_game"),
        #                 snap_pct=row.get("snap_pct")
        #             )
        #         ]
        #     )
            
        #     df = pd.concat([df, df_row], ignore_index=True)

        # self._bulk_upsert(NFLSeasonStats, records, conflict_column=("player_id", "season", "season_type"))
    
        print(f"{len(records)} seasonal stat rows ingested.")
        return df

def ingest_snap_counts(years: list[int]):
        """
        Weekly offensive snap counts. Useful for tracking role stability
        and detecting depth chart movement mid-season.
        """
        print(f"Ingesting weekly snap counts for {years}...")
        try:
            raw = nfl.import_snap_counts(years)
            raw = raw[raw["position"].isin(SKILL_POSITIONS)].copy()
        except Exception as e:
            print(f"Snap counts failed: {e}")
            return

        records = [
            NFLWeeklySnaps(
                player_id=str(row.get("pfr_player_id", "")),
                season=int(row["season"]),
                week=int(row["week"]),
                game_id=row.get("game_id", ""),
                team=row.get("team"),
                offense_snaps=_safe_int(row.get("offense_snaps")),
                offense_pct=row.get("offense_pct"),
                defense_snaps=_safe_int(row.get("defense_snaps")),
                st_snaps=_safe_int(row.get("st_snaps")),
            )
            for _, row in raw.iterrows()
        ]

        # self._bulk_upsert(NFLWeeklySnaps, records, conflict_column=None)  # no unique constraint here
        print(f"{len(records)} snap count rows ingested.")

In [127]:
# years = [2023, 2024]
years = list(range(2024, 2025)) #[2015, 2024]
raw = ingest_seasonal_stats(years)
# raw = ingest_snap_counts(years)

Ingesting seasonal stats for [2024]...
'game_id'
Data not available for 2024
Could not load team targets: 'pass_attempt'


In [120]:
display(raw)

,player_id,season,season_type,completions,attempts,passing_yards,passing_tds,interceptions,sacks,sack_yards,sack_fumbles,sack_fumbles_lost,passing_air_yards,passing_yards_after_catch,passing_first_downs,passing_epa,passing_2pt_conversions,pacr,dakota,carries,rushing_yards,rushing_tds,rushing_fumbles,rushing_fumbles_lost,rushing_first_downs,rushing_epa,rushing_2pt_conversions,receptions,targets,receiving_yards,receiving_tds,receiving_fumbles,receiving_fumbles_lost,receiving_air_yards,receiving_yards_after_catch,receiving_first_downs,receiving_epa,receiving_2pt_conversions,racr,target_share,air_yards_share,wopr_x,special_teams_tds,fantasy_points,fantasy_points_ppr,games,tgt_sh,ay_sh,yac_sh,wopr_y,ry_sh,rtd_sh,rfd_sh,rtdfd_sh,dom,w8dom,yptmpa,ppr_sh,snap_pct,wopr,tgt_per_game,catch_rate,fantasy_ppg_ppr,fantasy_ppg_half,completion_pct
0,00-0023459,2024,REG,368,584,3897.0,28,11.0,40.0,302.0,5,2,4014.0,2119.0,192.0,10.130993,2,17.796653,1.367085,22,107.0,0,0.0,0.0,7.0,4.280564,0,0,0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.000000,0,0.000000,0.000000,0.000000,0.000000,0.0,256.58,256.58,17,0.000000,0.000000,0.000000,0.000000,0.000000,0.00,0.000000,0.000000,0.000000,0.000000,0.000000,0.177032,NaN,0.000000,0.000000,NaN,15.092941,15.092941,0.630137
1,00-0024243,2024,REG,0,0,0.0,0,0.0,0.0,0.0,0,0,0.0,0.0,0.0,0.000000,0,0.000000,0.000000,0,0.0,0,0.0,0.0,0.0,0.000000,0,1,1,2.0,0,0.0,0.0,1.0,1.0,1.0,0.064605,0,2.000000,0.038462,0.009709,0.064488,0.0,0.20,1.20,1,0.034483,0.009434,0.006757,0.059271,0.008850,0.00,0.071429,0.055556,0.004425,0.007080,0.068966,0.010541,NaN,0.064488,1.000000,1.000000,1.200000,0.200000,NaN
2,00-0026158,2024,REG,162,248,1761.0,12,7.0,18.0,123.0,3,3,2215.0,651.0,87.0,2.999386,0,5.588957,0.707555,9,26.0,0,1.0,1.0,2.0,-7.002137,0,0,0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.000000,0,0.000000,0.000000,0.000000,0.000000,0.0,99.04,99.04,7,0.000000,0.000000,0.000000,0.000000,0.000000,0.00,0.000000,0.000000,0.000000,0.000000,0.000000,0.156155,NaN,0.000000,0.000000,NaN,14.148571,14.148571,0.653226
3,00-0026300,2024,REG,2,3,17.0,0,0.0,1.0,5.0,0,0,53.0,2.0,1.0,-1.809736,0,0.301887,0.000000,4,1.0,0,0.0,0.0,0.0,-3.562831,0,0,0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.000000,0,0.000000,0.000000,0.000000,0.000000,0.0,0.78,0.78,4,0.000000,0.000000,0.000000,0.000000,0.000000,0.00,0.000000,0.000000,0.000000,0.000000,0.000000,0.001842,NaN,0.000000,0.000000,NaN,0.195000,0.195000,0.666667
4,00-0026498,2024,REG,340,517,3762.0,20,8.0,28.0,213.0,4,2,3869.0,1726.0,176.0,34.734750,0,16.494773,1.607646,30,41.0,0,2.0,0.0,10.0,-3.011843,0,0,0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.000000,0,0.000000,0.000000,0.000000,0.000000,0.0,214.58,214.58,16,0.000000,0.000000,0.000000,0.000000,0.000000,0.00,0.000000,0.000000,0.000000,0.000000,0.000000,0.164457,NaN,0.000000,0.000000,NaN,13.411250,13.411250,0.657640
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
602,00-0039917,2024,REG,61,105,775.0,3,3.0,4.0,14.0,2,0,1073.0,271.0,35.0,14.553117,0,7.686124,0.250735,7,11.0,1,0.0,0.0,1.0,0.585619,0,0,0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.000000,0,0.000000,0.000000,0.000000,0.000000,0.0,44.10,44.10,5,0.000000,0.000000,0.000000,0.000000,0.000000,0.00,0.000000,0.000000,0.000000,0.000000,0.000000,0.107378,NaN,0.000000,0.000000,NaN,8.820000,8.820000,0.580952
603,00-0039918,2024,REG,351,562,3541.0,20,6.0,68.0,466.0,7,3,4486.0,1866.0,171.0,-43.804843,3,15.246817,1.290656,81,489.0,0,3.0,2.0,27.0,11.632084,0,0,0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.000000,0,0.000000,0.000000,0.000000,0.000000,0.0,254.54,254.54,17,0.000000,0.000000,0.000000,0.000000,0.000000,0.00,0.000000,0.000000,0.000000,0.000000,0.000000,0.196150,NaN,0.000000,0.000000,NaN,14.972941,14.972941,0.624555
604,00-0039919,2024,REG,0,0,0.0,0,0.0,0.0,0.0,0,0,0.0,0.0,0.0,0.000000,0,0.000000,0.000000,3,15.0,0,0.0,0.0,1.0,0.575859,0,54,101,734.0,3,2.0,1.0,1398.0,253.0

In [119]:
# display(
#     raw
#     .groupby('player_id')
#     .size()
#     .sort_values(ascending=False)
# )

(raw
    .groupby('player_id')
    .size()
    .sort_values(ascending=False))

player_id
00-0023459    1
00-0037834    1
00-0037838    1
00-0037840    1
00-0038040    1
             ..
00-0035662    1
00-0035676    1
00-0035685    1
00-0035689    1
00-0039921    1
Length: 607, dtype: int64

In [121]:
display(raw[raw['player_id']== '00-0034386'])

,player_id,season,season_type,completions,attempts,passing_yards,passing_tds,interceptions,sacks,sack_yards,sack_fumbles,sack_fumbles_lost,passing_air_yards,passing_yards_after_catch,passing_first_downs,passing_epa,passing_2pt_conversions,pacr,dakota,carries,rushing_yards,rushing_tds,rushing_fumbles,rushing_fumbles_lost,rushing_first_downs,rushing_epa,rushing_2pt_conversions,receptions,targets,receiving_yards,receiving_tds,receiving_fumbles,receiving_fumbles_lost,receiving_air_yards,receiving_yards_after_catch,receiving_first_downs,receiving_epa,receiving_2pt_conversions,racr,target_share,air_yards_share,wopr_x,special_teams_tds,fantasy_points,fantasy_points_ppr,games,tgt_sh,ay_sh,yac_sh,wopr_y,ry_sh,rtd_sh,rfd_sh,rtdfd_sh,dom,w8dom,yptmpa,ppr_sh,snap_pct,wopr,tgt_per_game,catch_rate,fantasy_ppg_ppr,fantasy_ppg_half,completion_pct
130,00-0034386,2024,REG,0,0,0.0,0,0.0,0.0,0.0,0,0,0.0,0.0,0.0,0.0,0,0.0,0.0,0,0.0,0,0.0,0.0,0.0,0.0,0,22,32,289.0,2,0.0,0.0,486.0,68.0,13.0,19.086959,0,13.165941,1.024191,2.237225,3.102344,0.0,40.9,62.9,15,0.061069,0.143617,0.034154,0.206497,0.082454,0.08,0.067358,0.068807,0.081227,0.081963,0.551527,0.048377,NaN,3.102344,2.133333,0.6875,4.193333,2.726667,NaN


In [122]:
raw = nfl.import_snap_counts(years)
raw = raw[raw["position"].isin(SKILL_POSITIONS)].copy()

In [67]:
display(raw)

,game_id,pfr_game_id,season,game_type,week,player,pfr_player_id,position,team,opponent,offense_snaps,offense_pct,defense_snaps,defense_pct,st_snaps,st_pct
4,2015_01_BAL_DEN,201509130den,2015,REG,1,Peyton Manning,MannPe00,QB,DEN,BAL,70.0,1.00,0.0,0.0,0.0,0.00
6,2015_01_BAL_DEN,201509130den,2015,REG,1,Emmanuel Sanders,SandEm00,WR,DEN,BAL,65.0,0.93,0.0,0.0,6.0,0.21
7,2015_01_BAL_DEN,201509130den,2015,REG,1,Demaryius Thomas,ThomDe03,WR,DEN,BAL,61.0,0.87,0.0,0.0,0.0,0.00
8,2015_01_BAL_DEN,201509130den,2015,REG,1,Owen Daniels,DaniOw00,TE,DEN,BAL,61.0,0.87,0.0,0.0,0.0,0.00
9,2015_01_BAL_DEN,201509130den,2015,REG,1,C.J. Anderson,AndeC.00,RB,DEN,BAL,52.0,0.74,0.0,0.0,0.0,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
26585,2024_22_KC_PHI,202502090phi,2024,SB,22,Kareem Hunt,HuntKa00,RB,KC,PHI,11.0,0.20,0.0,0.0,0.0,0.00
26586,2024_22_KC_PHI,202502090phi,2024,SB,22,Justin Watson,WatsJu01,WR,KC,PHI,10.0,0.18,0.0,0.0,8.0,0.28
26608,2024_22_KC_PHI,202502090phi,2024,SB,22,Peyton Hendershot,HendPe01,TE,KC,PHI,0.0,0.00,0.0,0.0,19.0,0.66
26609,2024_22_KC_PHI,202502090phi,2024,SB,22,Nikko Remigio,RemiNi00,WR,KC,PHI,0.0,0.00,0.0,0.0,13.0,0.45
